In [1]:
import matplotlib.pyplot as plt
import math
from math import log
import pandas as pd
import numpy as np
import random

### Load data

#### MovieLens

In [2]:
df_links = pd.read_csv('data/movielens/ml-latest-small/links.csv')
df_movies = pd.read_csv('data/movielens/ml-latest-small/movies.csv')
df_ratings = pd.read_csv('data/movielens/ml-latest-small/ratings.csv')
df_tags = pd.read_csv('data/movielens/ml-latest-small/tags.csv')

In [3]:
df_tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


#### Instacart

In [4]:
#instacart
df_products = pd.read_csv('data/instacart/products.csv')
df_carts_prior = pd.read_csv('data/instacart/order_products__prior.csv')
df_carts_train = pd.read_csv('data/instacart/order_products__train.csv')
df_carts = pd.concat([df_carts_prior, df_carts_train])
df_carts.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [5]:
df_carts.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [6]:
# transforms the data into a long list of lists, each containing product IDs 
# that were purchased in the same order
carts = df_carts[['order_id', 'product_id']].groupby('order_id')['product_id'].apply(list).to_list()
carts[0:3]

[[49302, 11109, 10246, 49683, 43633, 13176, 47209, 22035],
 [33120, 28985, 9327, 45918, 30035, 17794, 40141, 1819, 43668],
 [33754, 24838, 17704, 21903, 17668, 46667, 17461, 32665]]

### Ranking

Recommend 10 movies based on recent ratings

Use techniques to make sure the recommendation is reliable

In [7]:
... code here ...

SyntaxError: invalid syntax (215105830.py, line 1)

### Association rule mining

Calculate the number of frequent itemsets with varying levels for support

Try to guess what value of minimum support would be reasonable

Calculate association rules and find the one whose subsequent item has the least support (the one more in the tail)

#### Priori (Apyori)

In [9]:
!pip install apyori

Using legacy 'setup.py install' for apyori, since package 'wheel' is not installed.
    Running setup.py install for apyori ... done
You should consider upgrading via the '/Users/jonasson/.pyenv/versions/datascience-in-prod/bin/python3 -m pip install --upgrade pip' command.


In [10]:
from apyori import apriori

In [13]:
association_rules = apriori(carts, min_support=0.05, 
                            min_confidence=0.0,
                            min_lift=0.0, min_length=0)
association_rules = list(association_rules)

In [14]:
association_rules

[RelationRecord(items=frozenset({13176}), support=0.11802755639952744, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({13176}), confidence=0.11802755639952744, lift=1.0)]),
 RelationRecord(items=frozenset({21137}), support=0.08235808854711614, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({21137}), confidence=0.08235808854711614, lift=1.0)]),
 RelationRecord(items=frozenset({21903}), support=0.07522377657697074, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({21903}), confidence=0.07522377657697074, lift=1.0)]),
 RelationRecord(items=frozenset({24852}), support=0.14682570635575987, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({24852}), confidence=0.14682570635575987, lift=1.0)]),
 RelationRecord(items=frozenset({47209}), support=0.06601061599488117, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({47209}), confidence

In [15]:
idx = 5 #prints the 5th association rule

rule = association_rules[idx]
frequent_itemset = rule.items
support = rule.support

antecedent = rule.ordered_statistics[0].items_base
antecedent = [df_products.iloc[a-1]['product_name'] for a in antecedent]
consequent = rule.ordered_statistics[0].items_add
consequent = [df_products.iloc[c-1]['product_name'] for c in consequent]
lift = rule.ordered_statistics[0].lift
confidence = rule.ordered_statistics[0].confidence

print(f'{antecedent}->{consequent}')
print(f'support = {support}')
print(f'confidence = {confidence}')
print(f'lift = {lift}')

[]->['Organic Avocado']
support = 0.05505661395727482
confidence = 0.05505661395727482
lift = 1.0


#### FP-growth (mlxtend)

In [17]:
!pip install mlxtend

     |████████████████████████████████| 1.4 MB 1.9 MB/s eta 0:00:01
     |████████████████████████████████| 1.4 MB 7.9 MB/s eta 0:00:01
You should consider upgrading via the '/Users/jonasson/.pyenv/versions/datascience-in-prod/bin/python3 -m pip install --upgrade pip' command.


In [18]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth
from mlxtend.frequent_patterns import association_rules

In [19]:
# encode the dataset into a orders x items binary sparse matrix
te = TransactionEncoder()
te_data = te.fit(carts).transform(carts, sparse=True)
df = pd.DataFrame.sparse.from_spmatrix(te_data, columns=te.columns_)
# product indices must either start from 0 or be strings
df.columns = [str(i) for i in df.columns] 
# alternatively, reduce ids by 1
#carts_modified = [[carts[l][i]-1 for i in range(0, len(carts[l]))] for l in range(0, len(carts))]

/var/folders/5p/hd3qyvgs2yj3fc69b966hnfc0000gn/T/ipykernel_56892/133132032.py:4: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  df = pd.DataFrame.sparse.from_spmatrix(te_data, columns=te.columns_)


In [ ]:
frequent_itemsets = fpgrowth(df, min_support=0.05, use_colnames=True, verbose=1)

In [ ]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=xxxx)
rules